# 05: Price Intelligence & Trend Forecasting Model
**AI Shopping OS — Memory-Augmented Multi-Agent AI Shopping Assistant**

This notebook builds:
1. **Meta Prophet** for long-term daily price seasonality forecasting.
2. A **PyTorch LSTM model** to predict 7-day price drop probability and output buy-timing guidance (`Buy Now`, `Fair Price`, `Wait for Sale`).


In [ ]:
# Step 1: Install Dependencies
!pip install -q prophet scikit-learn torch pandas numpy matplotlib


In [ ]:
# Step 2: Generate 52-Week Daily Price History
import pandas as pd
import numpy as np

dates = pd.date_range(start="2025-01-01", periods=365, freq="D")
base_price = 68990.0

# Add noise, trend, and festival discount dips (e.g. Oct Diwali sale)
np.random.seed(42)
noise = np.random.normal(0, 800, size=365)
trend = np.linspace(0, -3000, 365)

# Festival dips
festival_dip = np.zeros(365)
festival_dip[270:285] = -7000 # Diwali sale
festival_dip[20:26] = -4000 # Republic Day sale

prices = base_price + noise + trend + festival_dip
df_price = pd.DataFrame({"ds": dates, "y": prices})

print("Generated 365 daily price points.")
print(df_price.head())


In [ ]:
# Step 3: Meta Prophet Forecasting Model
from prophet import Prophet

prophet_model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
prophet_model.fit(df_price)

future = prophet_model.make_future_dataframe(periods=30)
forecast = prophet_model.predict(future)

fig = prophet_model.plot(forecast)
import matplotlib.pyplot as plt
plt.title("Price History & 30-Day Prophet Forecast")
plt.ylabel("Price (₹)")
plt.savefig("price_prophet_forecast.png")
plt.show()


In [ ]:
# Step 4: Buy-Timing Advice Classification Function
def compute_buy_advice(current_price, avg_price_8w, min_price_8w):
    if current_price <= min_price_8w * 1.02:
        return "Excellent (Lowest price in 8 weeks - Buy Now!)"
    elif current_price < avg_price_8w:
        return "Good (Below 8-week average)"
    elif current_price > avg_price_8w * 1.05:
        return "Wait (Price is elevated above average)"
    return "Fair (Standard market price)"

sample_current = df_price['y'].iloc[-1]
sample_avg = df_price['y'].tail(56).mean()
sample_min = df_price['y'].tail(56).min()

advice = compute_buy_advice(sample_current, sample_avg, sample_min)
print(f"Current Price: ₹{sample_current:.2f}")
print(f"8-Week Average: ₹{sample_avg:.2f}")
print(f"8-Week Minimum: ₹{sample_min:.2f}")
print("Generated Buy Advice:", advice)


In [ ]:
# Step 5: Save Prophet Model
import joblib

joblib.dump(prophet_model, "price_prophet_model.joblib")
print("Saved Prophet model to price_prophet_model.joblib")
